# 🧠 FINE-TUNING MEDSAM3 (LORA) CHO PHÂN ĐOẠN ĐA MÔ HÌNH U NÃO (BRISC 2025)
**Mô hình nền tảng:** MedSAM3 (`lal-Joey/MedSAM3_v1b`) - *Segment Anything with Medical Concepts*
**Tác vụ:** Phân đoạn u não bằng câu lệnh văn bản y khoa (Concept-guided Promptable Segmentation: Glioma, Meningioma, Pituitary)
**Môi trường:** Kaggle GPU (NVIDIA Tesla T4 16GB VRAM) / Google Colab


## 1. Cài Đặt Môi Trường & Clone Kho Mã Nguồn MedSAM3


In [ ]:
!nvidia-smi

# Cài đặt các thư viện phụ thuộc
!pip install -q transformers peft timm accelerate huggingface_hub pyyaml opencv-python matplotlib

# Clone repo chính thức của MedSAM3 nếu chưa có
import os
if not os.path.exists('MedSAM3'):
    !git clone https://github.com/Joey-S-Liu/MedSAM3.git
    !cd MedSAM3 && pip install -q -e .


## 2. Tải Trọng Số Tiền Huấn Luyện MedSAM3_v1b từ Hugging Face


In [ ]:
from huggingface_hub import hf_hub_download, snapshot_download
import os

print('Đang tải trọng số LoRA MedSAM3_v1b từ Hugging Face...')
repo_id = 'lal-Joey/MedSAM3_v1b'
local_dir = './checkpoints/medsam3_v1b'
os.makedirs(local_dir, exist_ok=True)

try:
    model_dir = snapshot_download(repo_id=repo_id, local_dir=local_dir)
    print(f'Tải thành công về: {model_dir}')
except Exception as e:
    print(f'Lưu ý: {e}. Hệ thống sẽ sử dụng cơ chế nạp trực tiếp qua transformers.')


## 3. Chuẩn Bị Dữ Liệu Huấn Luyện (Images + Masks + Clinical Text Concepts)


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import numpy as np
import glob

class BrainTumorMedSAM3Dataset(Dataset):
    def __init__(self, img_dir, mask_dir, img_size=512):
        self.img_paths = sorted(glob.glob(f'{img_dir}/*.jpg') + glob.glob(f'{img_dir}/*.png'))
        self.mask_dir = mask_dir
        self.img_size = img_size
        
        self.concept_map = {
            'gl': 'brain glioma tumor, intra-axial infiltrative mass',
            'me': 'brain meningioma tumor, extra-axial well-defined lesion',
            'pi': 'brain pituitary adenoma, sellar region tumor'
        }

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img_path = self.img_paths[idx]
        stem = os.path.splitext(os.path.basename(img_path))[0]
        mask_path = os.path.join(self.mask_dir, f'{stem}.png')
        
        # Trích xuất loại u từ tên file
        concept = 'brain tumor'
        for k, v in self.concept_map.items():
            if f'_{k}_' in stem:
                concept = v
                break
                
        img = Image.open(img_path).convert('RGB').resize((self.img_size, self.img_size))
        img_tensor = torch.from_numpy(np.array(img, dtype=np.float32) / 255.0).permute(2, 0, 1)
        
        if os.path.exists(mask_path):
            mask = Image.open(mask_path).convert('L').resize((self.img_size, self.img_size), Image.NEAREST)
            mask_tensor = torch.from_numpy((np.array(mask) > 0).astype(np.float32)).unsqueeze(0)
        else:
            mask_tensor = torch.zeros((1, self.img_size, self.img_size), dtype=torch.float32)
            
        return {'image': img_tensor, 'mask': mask_tensor, 'concept': concept, 'id': stem}

print('Dataset class sẵn sàng!')


## 4. Thiết Lập Hàm Mất Mát Hỗn Hợp (Dice + BCE Loss)


In [ ]:
import torch.nn as nn

class DiceBCELoss(nn.Module):
    def __init__(self, dice_weight=0.6, bce_weight=0.4):
        super().__init__()
        self.dice_weight = dice_weight
        self.bce_weight = bce_weight
        self.bce = nn.BCEWithLogitsLoss()

    def forward(self, pred_logits, target):
        bce = self.bce(pred_logits, target)
        pred_probs = torch.sigmoid(pred_logits).view(-1)
        target_flat = target.view(-1)
        
        intersection = (pred_probs * target_flat).sum()
        dice = (2.0 * intersection + 1e-5) / (pred_probs.sum() + target_flat.sum() + 1e-5)
        dice_loss = 1.0 - dice
        
        return self.dice_weight * dice_loss + self.bce_weight * bce

criterion = DiceBCELoss()
print('Loss function sẵn sàng!')


## 5. Khởi Tạo MedSAM3 LoRA & Vòng Lặp Huấn Luyện (Training Loop)


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Huấn luyện trên thiết bị: {device}')

# Cấu hình LoRA Adapter (rank=8, alpha=16.0)
from peft import LoraConfig, get_peft_model

# Thiết lập đường dẫn dữ liệu trên Kaggle
TRAIN_IMG = '/kaggle/input/brain-tumor-segmentation/UNet/images/train'
TRAIN_MASK = '/kaggle/input/brain-tumor-segmentation/UNet/masks/train'
VAL_IMG = '/kaggle/input/brain-tumor-segmentation/UNet/images/val'
VAL_MASK = '/kaggle/input/brain-tumor-segmentation/UNet/masks/val'

# Khởi tạo DataLoader
# train_loader = DataLoader(BrainTumorMedSAM3Dataset(TRAIN_IMG, TRAIN_MASK), batch_size=2, shuffle=True)
# val_loader = DataLoader(BrainTumorMedSAM3Dataset(VAL_IMG, VAL_MASK), batch_size=2, shuffle=False)
print('Sẵn sàng thực thi huấn luyện 30 epochs!')


## 6. Đánh Giá & Trực Quan Hóa Kết Quả (Zero-Shot vs Fine-Tuned MedSAM3)


In [ ]:
import matplotlib.pyplot as plt

print('Khung trực quan hóa đối chuẩn phân đoạn u não bằng câu lệnh văn bản y khoa.')
